In [ ]:
# Automated Setup: Install missing libraries and generate data if missing
import sys
import os
!{sys.executable} -m pip install -r requirements.txt

if not os.path.exists('fintech.db'):
    print("Database missing. Generating data now...")
    !{sys.executable} scripts/generate_data.py

# Fintech A/B Testing Analysis: Credit Health Dashboard

## 1. Project Context
**Objective**: This analysis evaluates the impact of the "Credit Health Dashboard" feature on credit card application rates. We aim to determine if users who see their credit health insights convert at a higher rate without increasing rejection risk.

**Recruiter Notes**:
- This analysis uses **raw event-stream data** extracted via SQL.
- Includes **Pre-test Power Analysis**, **Hypothesis Testing**, and **Practical Significance (Effect Size)**.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Hypothesis Formulation
We are measuring the **Application Conversion Rate** (Session Start to Application Submission).

- **$H_0$ (Null Hypothesis)**: The new Credit Health Dashboard has no effect on application conversion rates. ($p_{control} = p_{treatment}$)
- **$H_1$ (Alternative Hypothesis)**: The new Credit Health Dashboard increases application conversion rates. ($p_{treatment} > p_{control}$)

**Confidence Level**: 95% ($\alpha = 0.05$)

In [ ]:
# Load SQL Script and Extract Data
with open('sql/extract_metrics.sql', 'r') as f:
    query = f.read()

conn = sqlite3.connect('fintech.db')
df = pd.read_sql_query(query, conn)
conn.close()

print(f"Data Loaded: {len(df)} users analyzed.")
df.head()

## 3. Pre-Test Power Analysis
Before reaching conclusions, we determine if we have enough statistical power to detect a meaningful effect.

In [ ]:
effect_size = 0.05 # We want to detect at least a 5% relative lift
alpha = 0.05
power = 0.8

analysis = NormalIndPower()
required_n = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1)

actual_n_low = df[df['test_group'] == 'control'].shape[0]
actual_n_high = df[df['test_group'] == 'treatment'].shape[0]

print(f"Required Sample Size per Group: {int(required_n)}")
print(f"Actual Sample Size: Control={actual_n_low}, Treatment={actual_n_high}")

## 4. Significance Testing
We use a **Z-Test for Proportions** since we are comparing conversion rates.

In [ ]:
summary = df.groupby('test_group')['has_submitted_app'].agg(['sum', 'count', 'mean'])
summary.columns = ['conversions', 'total_users', 'conv_rate']

counts = summary['conversions'].values
nobs = summary['total_users'].values

z_stat, p_val = proportions_ztest(counts, nobs, alternative='smaller' if summary.index[0] == 'control' else 'larger')
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(counts, nobs, alpha=0.05)

print(f"Conversion Rates: Control={summary.loc['control', 'conv_rate']:.2%}, Treatment={summary.loc['treatment', 'conv_rate']:.2%}")
print(f"P-Value: {p_val:.4f}")
print(f"Confidence Interval Treatment: [{lower_treat:.2%}, {upper_treat:.2%}]")

## 5. Visualizing the Impact

In [ ]:
sns.barplot(x='test_group', y='has_submitted_app', data=df, ci=95, palette=['#D1D1D1', '#4CAF50'])
plt.title('Conversion Rate by Test Group (95% CI)')
plt.ylabel('Conversion Rate')
plt.show()